# Navier Stokes for Schäfer-Turek

In [ ]:
from ngsolve import *
import ngsolve.webgui as ngw
from ngsxditto import *
from netgen.occ import *


First we build the Geometry.

In [ ]:
shape = Rectangle(2,0.41).Circle(0.2,0.2,0.05).Reverse().Face()
shape.edges.name="wall"
shape.edges.Min(X).name="inlet"
shape.edges.Max(X).name="outlet"
mesh = Mesh(OCCGeometry(shape, dim=2).GenerateMesh(maxh=0.07)).Curve(3)
ngw.Draw(mesh);

We define some parameters and inflow boundary conditions.

In [ ]:
order = 4
fluid_params = FluidParameters(viscosity=1e-3)
uin = CoefficientFunction((1.5*4*y*(0.41-y)/(0.41*0.41), 0))

Now we can define our fluid discretization with our boundary conditions. Then we solve the Stokes problem to obtain our initial values.

In [ ]:
fluid = BDMDG(mesh, order=order, fluid_params=fluid_params)
dirichlet = {"inlet": uin, "cyl":CF((0, 0)), "wall": CF((0, 0))}
fluid.Initialize(dirichlet=dirichlet)
stokes = fluid.SolveStokes()
vel, pressure = stokes.components[0], stokes.components[1]
fluid.SetInitialValues(vel, pressure)
fluid.InitializeForms()

sceneu = ngw.Draw(fluid.gfu.components[0], min=0, max=2)
scenep = ngw.Draw(fluid.gfu.components[1])

In [ ]:
tend = 1
time = 0
while time <= tend:
    fluid.OneStep()
    sceneu.Redraw()
    scenep.Redraw()
    time += fluid.dt